In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# URL base del sitio
BASE_URL = 'https://books.toscrape.com'

def get_soup(url):
    """Obtiene el objeto BeautifulSoup de una URL"""
    response = requests.get(url)
    response.raise_for_status()
    return BeautifulSoup(response.content, 'html.parser')

def extract_rating(rating_class):
    """Convierte la clase de rating a número"""
    ratings = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    for key in ratings:
        if key in rating_class:
            return ratings[key]
    return None

def scrape_book_details(book_url, category_name):
    """Extrae detalles de un libro individual"""
    try:
        soup = get_soup(book_url)
        
        # Información básica
        title = soup.find('h1').text.strip()
        
        # Precio
        price = soup.find('p', class_='price_color').text.strip()
        price = float(price.replace('£', ''))
        
        # Rating
        rating_tag = soup.find('p', class_='star-rating')
        rating = extract_rating(rating_tag['class']) if rating_tag else None
        
        # Disponibilidad
        availability = soup.find('p', class_='instock availability').text.strip()
        
        # Descripción
        description_tag = soup.find('meta', attrs={'name': 'description'})
        description = description_tag['content'].strip() if description_tag else ''
        
        # UPC
        table = soup.find('table', class_='table table-striped')
        product_info = {}
        for row in table.find_all('tr'):
            header = row.find('th').text.strip()
            value = row.find('td').text.strip()
            product_info[header] = value
        
        upc = product_info.get('UPC', '')
        
        return {
            'title': title,
            'price': price,
            'rating': rating,
            'availability': availability,
            'description': description,
            'upc': upc,
            'category': category_name,
            'url': book_url
        }
    except Exception as e:
        print(f"✗ Error en {book_url}: {e}")
        return None

def get_all_book_urls_from_category(category_url, category_name):
    """Obtiene todas las URLs de libros de una categoría (incluyendo paginación)"""
    book_urls = []
    current_url = category_url
    
    while current_url:
        soup = get_soup(current_url)
        
        # Extraer URLs de libros
        book_containers = soup.find_all('article', class_='product_pod')
        
        for book in book_containers:
            book_link = book.find('h3').find('a')['href']
            book_url = BASE_URL + '/catalogue/' + book_link.replace('../../../', '')
            book_urls.append((book_url, category_name))
        
        # Buscar siguiente página
        next_button = soup.find('li', class_='next')
        if next_button:
            next_page = next_button.find('a')['href']
            if 'catalogue/category' in current_url:
                current_url = '/'.join(current_url.split('/')[:-1]) + '/' + next_page
            else:
                current_url = category_url.replace('index.html', next_page)
        else:
            current_url = None
    
    return book_urls

def scrape_all_categories_parallel():
    """Scrapea todas las categorías y sus libros EN PARALELO"""
    print("🕷️  Iniciando scraping RÁPIDO de Books to Scrape...\n")
    
    start_time = time.time()
    
    # Obtener todas las categorías
    soup = get_soup(BASE_URL)
    category_list = soup.find('ul', class_='nav nav-list').find('ul').find_all('a')
    
    print(f"📂 {len(category_list)} categorías encontradas\n")
    
    # Paso 1: Obtener todas las URLs de libros (todavía secuencial, pero rápido)
    print("📋 Recolectando URLs de libros...")
    all_book_urls = []
    
    for i, category in enumerate(category_list, 1):
        category_name = category.text.strip()
        category_url = BASE_URL + '/' + category['href']
        
        print(f"  [{i}/{len(category_list)}] {category_name}...", end=' ')
        book_urls = get_all_book_urls_from_category(category_url, category_name)
        all_book_urls.extend(book_urls)
        print(f"{len(book_urls)} libros")
    
    print(f"\n📚 Total de libros a scrapear: {len(all_book_urls)}\n")
    
    # Paso 2: Scrapear todos los libros EN PARALELO
    print("⚡ Scrapeando libros en paralelo...\n")
    all_books = []
    
    # Usar ThreadPoolExecutor para paralelizar
    with ThreadPoolExecutor(max_workers=20) as executor:
        # Enviar todas las tareas
        future_to_url = {
            executor.submit(scrape_book_details, url, cat): (url, cat) 
            for url, cat in all_book_urls
        }
        
        # Procesar resultados conforme se completan
        completed = 0
        for future in as_completed(future_to_url):
            completed += 1
            book_data = future.result()
            
            if book_data:
                all_books.append(book_data)
            
            # Mostrar progreso cada 50 libros
            if completed % 50 == 0:
                print(f"  ✓ {completed}/{len(all_book_urls)} libros procesados...")
    
    elapsed_time = time.time() - start_time
    
    print(f"\n✅ Scraping completado en {elapsed_time:.2f} segundos")
    print(f"📊 {len(all_books)} libros scrapeados exitosamente\n")
    
    return all_books

# Ejecutar el scraping
books_data = scrape_all_categories_parallel()

# Convertir a DataFrame
df_books = pd.DataFrame(books_data)

# Mostrar estadísticas
print("📈 Estadísticas:")
print(f"  - Total libros: {len(df_books)}")
print(f"  - Categorías únicas: {df_books['category'].nunique()}")
print(f"  - Precio promedio: £{df_books['price'].mean():.2f}")
print(f"  - Rating promedio: {df_books['rating'].mean():.2f} estrellas")

print("\n📋 Primeros 5 libros:")
print(df_books[['title', 'price', 'rating', 'category']].head())

# Guardar a CSV
df_books.to_csv('books_scraped.csv', index=False)
print("\n💾 Datos guardados en 'books_scraped.csv'")

🕷️  Iniciando scraping RÁPIDO de Books to Scrape...

📂 50 categorías encontradas

📋 Recolectando URLs de libros...
  [1/50] Travel... 11 libros
  [2/50] Mystery... 32 libros
  [3/50] Historical Fiction... 26 libros
  [4/50] Sequential Art... 75 libros
  [5/50] Classics... 19 libros
  [6/50] Philosophy... 11 libros
  [7/50] Romance... 35 libros
  [8/50] Womens Fiction... 17 libros
  [9/50] Fiction... 65 libros
  [10/50] Childrens... 29 libros
  [11/50] Religion... 7 libros
  [12/50] Nonfiction... 110 libros
  [13/50] Music... 13 libros
  [14/50] Default... 152 libros
  [15/50] Science Fiction... 16 libros
  [16/50] Sports and Games... 5 libros
  [17/50] Add a comment... 67 libros
  [18/50] Fantasy... 48 libros
  [19/50] New Adult... 6 libros
  [20/50] Young Adult... 54 libros
  [21/50] Science... 14 libros
  [22/50] Poetry... 19 libros
  [23/50] Paranormal... 1 libros
  [24/50] Art... 8 libros
  [25/50] Psychology... 7 libros
  [26/50] Autobiography... 9 libros
  [27/50] Parenting... 1 